In [1]:
"EGARCH API"
from arch import arch_model
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

chart1 = pd.read_csv("S&P 500 Historical Data 1987.csv")
chart2 = pd.read_csv("S&P 500 Historical Data 2000-2002.csv")
chart3 = pd.read_csv("S&P 500 Historical Data 2008.csv")
chart4 = pd.read_csv("S&P 500 Historical Data 2020-2022.csv")

chart1['Date'] = pd.to_datetime(chart3['Date'], format='%m/%d/%Y')
month = chart1['Date'].dt.month
month = month.astype(int)

volatility = np.concatenate([
    chart1['Change %'].dropna().values,
    chart2['Change %'].dropna().values,
    chart3['Change %'].dropna().values,
    chart4['Change %'].dropna().values
])

volatility = pd.Series(volatility).str.rstrip('%').astype(float)

returns = volatility.values / 100.0
returns = returns - returns.mean()

returns_scaled = returns * 100

vol_proxy = np.log((returns ** 2) + 1e-6)

# -------------------------
# TRAIN / VAL SPLIT
# -------------------------
n = len(returns)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

# -------------------------
# EGARCH MODEL
# -------------------------
am = arch_model(
    returns_scaled,
    vol='EGARCH',
    p=1, o=1, q=1,
    dist='normal',
    mean='Zero'
)

res = am.fit(disp='off', last_obs=train_end)
print(res.summary())

# Apply trained params to full dataset
res_full = am.fix(res.params)

# Conditional volatility (sigma)
sigma = res_full.conditional_volatility / 100

# Convert to log variance (same scale as proxy)
egarch_log = np.log(sigma ** 2 + 1e-6)

# Align lengths
min_len = min(len(egarch_log), len(vol_proxy))
egarch_log = egarch_log[-min_len:]
vol_proxy = vol_proxy[-min_len:]

# -------------------------
# SPLIT DATA
# -------------------------
train_pred = egarch_log[:train_end]
val_pred   = egarch_log[train_end:val_end]
test_pred  = egarch_log[val_end:]

train_true = vol_proxy[:train_end]
val_true   = vol_proxy[train_end:val_end]
test_true  = vol_proxy[val_end:]

# -------------------------
# RMSE FUNCTION
# -------------------------
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

print("\nRMSE Results:")
print("Train RMSE:", rmse(train_true, train_pred))
print("Validation RMSE:", rmse(val_true, val_pred))
print("Test RMSE:", rmse(test_true, test_pred))

# -------------------------
# PLOT 1: TRUE vs PREDICTED
# -------------------------
plt.figure()
plt.plot(vol_proxy, label="True Proxy Volatility")
plt.plot(egarch_log, label="EGARCH Predicted")
plt.axvline(train_end, linestyle='--', label='Train Split')
plt.axvline(val_end, linestyle='--', label='Val Split')
plt.legend()
plt.title("True vs Predicted Proxy Volatility")
plt.show()

# -------------------------
# PLOT 2: PREDICTION RANGE COMPARISON
# -------------------------
plt.figure()
plt.hist(vol_proxy, bins=50, alpha=0.5, label="True")
plt.hist(egarch_log, bins=50, alpha=0.5, label="Predicted")
plt.legend()
plt.title("Distribution Comparison (Prediction Range)")
plt.show()



FileNotFoundError: [Errno 2] No such file or directory: 'S&P 500 Historical Data 1987.csv'